# Foreground Segmentation (Pre-Diffusion)

This notebook isolates the tank in each authentic image from its background,
producing a binary mask per image. These masks are used in the later
diffusion-augmentation step to constrain edits to the background, so that the
class-defining tank structure is preserved.

The pipeline combines two models:

- **GroundingDINO** performs open-vocabulary detection from the text prompt
  `"a tank."` and returns a bounding box around the vehicle.
- **SAM (Segment Anything Model)** takes that box as a prompt and produces a
  precise pixel-level mask.

We only process the `C1` camera-angle subset, and the run is resumable: images
that already have a mask are skipped. A final diagnostic step flags and removes
fragmented masks, which are typically caused by detection failures or
background clutter.

## 1. Setup

Install dependencies and download the SAM checkpoint. A GPU runtime is required;
on CPU the segmentation is impractically slow.

In [ ]:
import platform
import torch

print(f"System: {platform.system()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected")

!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q diffusers accelerate
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O /content/sam_vit_h_4b8939.pth

## 2. Mount Drive and set up paths

Images are organised by class under `augmentation_data`. For each class we read the originals and write masks and previews to sibling folders. `CLASS_NAME` here is only used for the quick single-class check below; the full run in Section 4 and below iterates over all three classes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE = '/content/drive/MyDrive/augmentation_data'
CLASS_NAME = 't72'   # switch to 't80' or 't90' for the other classes

ORIGINALS_DIR = f'{BASE}/{CLASS_NAME}'
MASKS_DIR = f'{BASE}/{CLASS_NAME}_masks'
PREVIEW_DIR = f'{BASE}/{CLASS_NAME}_preview'

for d in [MASKS_DIR, PREVIEW_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

image_files = sorted([f for f in os.listdir(ORIGINALS_DIR)
                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"Found {len(image_files)} images")
print(f"First 5: {image_files[:5]}")

## 3. Load the models

GroundingDINO and SAM are loaded once onto the GPU and reused for every image.
GroundingDINO provides the bounding box; SAM refines that box into a mask.

In [ ]:
import numpy as np
import cv2
import torch
from PIL import Image as PILImage
from segment_anything import sam_model_registry, SamPredictor
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# GroundingDINO provides the bounding box, SAM refines it into a mask
gdino_processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    "IDEA-Research/grounding-dino-tiny"
).to("cuda")

sam = sam_model_registry["vit_h"](checkpoint="/content/sam_vit_h_4b8939.pth")
sam.to(device="cuda")
predictor = SamPredictor(sam)

print("GroundingDINO + SAM loaded.")

## 4. Segmentation function

The function below handles a single image end to end. To keep VRAM usage
manageable, large images are downscaled to a maximum dimension of 1024 px before
detection and segmentation, and the resulting mask is upscaled back to the
original resolution with nearest-neighbour interpolation (which keeps the mask
binary).

The detection box is padded by 15% on each side so that SAM has some context
around the vehicle. After segmentation, masks covering an implausibly small or
large fraction of the frame are flagged, since these usually indicate a failed
detection. A side-by-side preview (original next to mask overlay) is saved for
visual inspection.

In [ ]:
# SAM runs at this resolution; large images are downscaled first to save VRAM.
MAX_DIM = 1024

def segment_tank(image_path, output_mask_path, output_preview_path, text_prompt="a tank."):
    image = cv2.imread(image_path)
    if image is None:
        print(f"  Could not load: {image_path}")
        return False
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    orig_h, orig_w = image_rgb.shape[:2]

    # Downscale for processing if the image is larger than MAX_DIM
    scale = min(MAX_DIM / orig_w, MAX_DIM / orig_h, 1.0)
    if scale < 1.0:
        work_w, work_h = int(orig_w * scale), int(orig_h * scale)
        image_small = cv2.resize(image_rgb, (work_w, work_h), interpolation=cv2.INTER_AREA)
    else:
        image_small = image_rgb
        work_w, work_h = orig_w, orig_h

    image_pil = PILImage.fromarray(image_small)

    # GroundingDINO locates the tank and returns candidate boxes
    inputs = gdino_processor(images=image_pil, text=text_prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = gdino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=0.20,
        text_threshold=0.15,
        target_sizes=[image_pil.size[::-1]],
    )[0]

    boxes = results["boxes"]
    scores = results["scores"]
    if len(boxes) == 0:
        print(f"  No tank found: {os.path.basename(image_path)}")
        return False

    # Keep the highest-scoring box and pad it by 15% on each side
    best_idx = scores.argmax().item()
    x1, y1, x2, y2 = boxes[best_idx].cpu().numpy().astype(int)
    box_w, box_h = x2 - x1, y2 - y1
    pad_x, pad_y = int(box_w * 0.15), int(box_h * 0.15)
    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(work_w, x2 + pad_x)
    y2 = min(work_h, y2 + pad_y)

    # SAM turns the box into a mask
    predictor.set_image(image_small)
    masks, _, _ = predictor.predict(box=np.array([x1, y1, x2, y2]), multimask_output=False)
    mask_small = masks[0]

    # Upscale the mask back to the original resolution
    if scale < 1.0:
        mask = cv2.resize(
            mask_small.astype(np.uint8), (orig_w, orig_h), interpolation=cv2.INTER_NEAREST
        ).astype(bool)
    else:
        mask = mask_small

    # Flag masks that cover suspiciously little or too much of the frame
    coverage = mask.sum() / (orig_h * orig_w)
    if coverage < 0.02 or coverage > 0.80:
        print(f"  Warning: mask coverage {coverage:.1%} - {os.path.basename(image_path)}")

    cv2.imwrite(output_mask_path, (mask * 255).astype(np.uint8))

    # Save a side-by-side preview: original next to mask overlay + detection box
    overlay = image_rgb.copy()
    overlay[mask] = overlay[mask] * 0.5 + np.array([255, 0, 0]) * 0.5
    if scale < 1.0:
        x1, y1 = int(x1 / scale), int(y1 / scale)
        x2, y2 = int(x2 / scale), int(y2 / scale)
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 3)
    preview = np.concatenate([image_rgb, overlay.astype(np.uint8)], axis=1)
    cv2.imwrite(output_preview_path, cv2.cvtColor(preview, cv2.COLOR_RGB2BGR))

    del inputs, outputs, masks, mask_small
    torch.cuda.empty_cache()
    return True

## 5. Run segmentation across all classes

This processes the `C1` subset for all three classes. Filenames encode metadata
in underscore-separated fields; field index 6 holds the camera angle, so we keep
only files where that field equals `C1`.

The loop is resumable: any image that already has a mask is skipped, so the run
can be interrupted and restarted without redoing work. To avoid filling Drive
with preview images, only roughly every 20th preview is kept; the rest are
written to a temporary path and discarded.

After SAM produces each mask, a light morphological closing fills small holes and
smooths the edges.

In [ ]:
import time

CLASSES = ['t72', 't80', 't90']


def filter_by_attribute(files, position, value):
    filtered = []
    for f in files:
        parts = f.split('_')
        if len(parts) > position and parts[position] == value:
            filtered.append(f)
    return filtered


def post_process_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return
    mask = (mask > 127).astype(np.uint8) * 255
    # Morphological closing fills small holes and smooths the edges
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = (mask > 127).astype(np.uint8) * 255
    cv2.imwrite(mask_path, mask)


stats = {}
total_start = time.time()

for class_idx, class_name in enumerate(CLASSES):
    print(f"\nClass {class_idx + 1}/3: {class_name.upper()}")

    originals_dir = f'{BASE}/{class_name}'
    masks_dir = f'{BASE}/{class_name}_masks'
    preview_dir = f'{BASE}/{class_name}_preview'
    Path(masks_dir).mkdir(parents=True, exist_ok=True)
    Path(preview_dir).mkdir(parents=True, exist_ok=True)

    all_files = sorted([f for f in os.listdir(originals_dir)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    c1_files = filter_by_attribute(all_files, position=6, value='C1')
    print(f"Total images: {len(all_files)}, C1 images: {len(c1_files)}")

    # Skip images that already have a mask so the run can be resumed
    already_done = {f.replace('_mask.png', '.JPG') for f in os.listdir(masks_dir)
                    if f.endswith('_mask.png')}
    to_process = [f for f in c1_files if f not in already_done]
    print(f"Already processed: {len(already_done)}, remaining: {len(to_process)}")

    if not to_process:
        print(f"All C1 images for {class_name} are already segmented, skipping.")
        stats[class_name] = {'total_c1': len(c1_files), 'segmented': 0, 'time': 0}
        continue

    class_start = time.time()
    segmented_count = 0
    preview_interval = max(1, len(to_process) // 20)

    for i, fname in enumerate(to_process):
        src = os.path.join(originals_dir, fname)
        mask_path = os.path.join(masks_dir, os.path.splitext(fname)[0] + '_mask.png')
        # Only keep every Nth preview to avoid filling Drive with preview images
        preview_path = (os.path.join(preview_dir, f"seg_{fname}")
                        if i % preview_interval == 0 else '/tmp/temp_preview.jpg')

        if segment_tank(src, mask_path, preview_path):
            post_process_mask(mask_path)
            segmented_count += 1

        if (i + 1) % 25 == 0 or (i + 1) == len(to_process):
            elapsed = time.time() - class_start
            rate = (i + 1) / elapsed
            eta = (len(to_process) - (i + 1)) / rate if rate > 0 else 0
            print(f"  [{i+1}/{len(to_process)}] segmented: {segmented_count}, "
                  f"{rate:.1f} img/s, ETA: {eta/60:.1f} min")

    class_time = time.time() - class_start
    stats[class_name] = {'total_c1': len(c1_files), 'segmented': segmented_count, 'time': class_time}
    print(f"{class_name.upper()} done in {class_time/60:.1f} min, segmented {segmented_count}")

total_time = time.time() - total_start
print(f"\nTotal time: {total_time/60:.1f} minutes")
for class_name, s in stats.items():
    print(f"{class_name.upper()}: {s['total_c1']} C1 images, "
          f"{s['segmented']} segmented this run, {s['time']/60:.1f} min")

## 6. Quality control: detect fragmented masks

A good mask is one connected region covering the tank. When detection or
segmentation fails (for example on cluttered backgrounds), the mask can break
into many small disconnected pieces instead. The check below counts connected
components larger than 100 px and flags any mask with more than 20 of them as
fragmented, so they can be inspected and removed before the diffusion step.

In [ ]:
def detect_fragmented_mask(mask_path, threshold_components=20):
    """Return True if the mask is fragmented into many small components
    rather than one connected blob."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return False, 0, 0

    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    num_labels, labels, comp_stats, _ = cv2.connectedComponentsWithStats(binary)

    # Count only components large enough to not be noise
    significant = sum(1 for i in range(1, num_labels)
                      if comp_stats[i, cv2.CC_STAT_AREA] > 100)
    coverage = (binary > 0).sum() / binary.size

    return significant > threshold_components, significant, coverage


fragmented = {cls: [] for cls in CLASSES}
for cls in CLASSES:
    masks_dir = f'{BASE}/{cls}_masks'
    if not os.path.exists(masks_dir):
        continue

    mask_files = [f for f in os.listdir(masks_dir) if f.endswith('_mask.png')]
    print(f"Checking {len(mask_files)} masks for {cls}...")
    for mf in mask_files:
        is_frag, n_comp, cov = detect_fragmented_mask(os.path.join(masks_dir, mf))
        if is_frag:
            fragmented[cls].append((mf, n_comp, cov))

print("\nFragmented masks:")
for cls in CLASSES:
    print(f"\n{cls.upper()}: {len(fragmented[cls])} fragmented")
    for mf, n_comp, cov in fragmented[cls][:5]:
        print(f"  {mf}: {n_comp} components, {cov:.1%} coverage")
    if len(fragmented[cls]) > 5:
        print(f"  ... and {len(fragmented[cls]) - 5} more")

## 7. Remove fragmented masks

The flagged masks are deleted permanently. They are not re-processed in this run;
if needed, they can be regenerated by re-running Section 5, which will treat the
now-missing masks as unprocessed.

In [ ]:
deleted_count = 0
for cls in CLASSES:
    masks_dir = f'{BASE}/{cls}_masks'
    for mf, _, _ in fragmented[cls]:
        mask_path = os.path.join(masks_dir, mf)
        if os.path.exists(mask_path):
            os.remove(mask_path)
            deleted_count += 1

print(f"Deleted {deleted_count} fragmented masks permanently")